# 🏗️ Day 3B — Few-Shot CoT + Structured JSON Output
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 Where We Are in the Story

| Day | What we built | Gap left open |
|-----|--------------|---------------|
| Day 2 | ReAct agent — short-term memory, tool calls | Reasoning was implicit, uncontrolled |
| Day 3A | Added CoT to system prompt — reasoning visible | Output was still prose — not machine-parseable |
| **Day 3B** | **Few-shot examples + JSON schema** | **Lock in format so every response is a validated object** |

**The core problem we solve today:**
Your tax pipeline needs to extract `{category, section, rate, reasoning, confidence}` from every agent response.
Well-reasoned prose is great for humans. It crashes Python code that expects a dict.

**The solution:** Provide 3 worked examples showing *exactly* the JSON we expect.
The model sees the pattern and replicates it for every new input.

---

## 🎯 Learning Objectives
1. Write 3 few-shot CoT examples that lock in a JSON output format
2. Define a `TaxClassification` Pydantic model for response validation
3. Build a robust JSON extractor that handles model response variations
4. Run on 10 Indian tax scenarios and verify parse success rate
5. Implement confidence gating to flag ambiguous cases for human review

---
## ⏱️ Time: 45 minutes

---
## ⚙️ Step 1: Install & Import

In [ ]:
%pip install openai pydantic --quiet

In [ ]:
import os
import re
import json
import datetime
from typing import Literal
from openai import AzureOpenAI
from getpass import getpass
from pydantic import BaseModel, Field, field_validator, ValidationError

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI Client

In [ ]:
AZURE_OPENAI_ENDPOINT = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY  = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME = input("Deployment name (e.g. gpt-4o): ").strip()

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = "2024-08-01-preview",
    timeout        = 120.0,
    max_retries    = 3
)
print("✅ Azure OpenAI client initialised")

---
## 🧱 Step 3: Define the Output Schema with Pydantic

This is the **contract** between your LLM and your downstream tax pipeline.
Every field has a type, description, and validation rule.
If the model returns garbage, Pydantic raises a `ValidationError` — your pipeline never sees bad data.

In [ ]:
class TaxClassification(BaseModel):
    """Structured output for a GST/tax scenario classification."""

    category: str = Field(
        description=(
            "Transaction type, e.g.: B2B Intra-State, B2B Inter-State, "
            "Export of Services, Import of Services RCM, Exempt Supply, "
            "Composition Scheme, E-Commerce Supply"
        )
    )
    applicable_tax: Literal[
        "CGST+SGST", "IGST", "Customs+IGST", "Exempt", "Zero-Rated", "Not Applicable"
    ] = Field(
        description="Which tax head applies to this transaction"
    )
    rate: str = Field(
        description="Tax rate as a string, e.g. '18%', '0% (LUT filed)', 'Exempt'"
    )
    section: str = Field(
        description="Primary CGST/IGST/Customs Act section that governs this scenario"
    )
    reasoning: str = Field(
        description=(
            "4-step reasoning: "
            "CLASSIFY the transaction, LOCATE the applicable rule, "
            "APPLY it to the specific facts, state CONCLUSION."
        )
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence score 0.0–1.0. Below 0.80 means the case needs human review."
    )
    human_review_required: bool = Field(
        description="Set True if confidence < 0.80 or the scenario has ambiguous facts."
    )

    @field_validator("human_review_required", mode="before")
    @classmethod
    def auto_flag_low_confidence(cls, v, info):
        """Auto-raise the review flag whenever confidence is below 0.80."""
        confidence = info.data.get("confidence", 1.0)
        return True if confidence < 0.80 else bool(v)


# Print the schema so participants can see what we're asking the model to produce
print("📋 TaxClassification schema fields:")
print("-" * 60)
for fname, finfo in TaxClassification.model_fields.items():
    ftype = finfo.annotation.__name__ if hasattr(finfo.annotation, '__name__') else str(finfo.annotation)
    desc  = (finfo.description or "")[:55]
    print(f"  {fname:<26} {ftype:<12}  {desc}")

# Build a compact schema string to inject into the system prompt
SCHEMA_DESCRIPTION = json.dumps({
    "category":             "string — transaction type",
    "applicable_tax":       "one of: CGST+SGST | IGST | Customs+IGST | Exempt | Zero-Rated | Not Applicable",
    "rate":                 "string — e.g. '18%' or '0% (LUT filed)'",
    "section":              "string — primary governing section",
    "reasoning":            "string — CLASSIFY / LOCATE RULE / APPLY / CONCLUSION",
    "confidence":           "float 0.0–1.0",
    "human_review_required":"boolean",
}, indent=2)

print()
print("✅ Schema defined and ready to inject into prompt")

---
## 🔧 Step 4: JSON Extractor — Handles Messy Model Responses

Even when we ask the model to return only JSON, it sometimes wraps it in markdown fences (` ```json `).
This helper strips the fences, parses, and validates the response through Pydantic.

In [ ]:
def extract_and_validate(raw_text):
    """
    Extract JSON from model response and validate against TaxClassification.
    Returns (TaxClassification | None, error_message | None).
    """
    # 1. Strip markdown fences if present
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw_text.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r"```$", "", cleaned.strip(), flags=re.MULTILINE).strip()

    # 2. Parse JSON
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"

    # 3. Validate with Pydantic
    try:
        result = TaxClassification.model_validate(data)
        return result, None
    except ValidationError as e:
        return None, f"Validation error: {e.errors()[0]['msg']} (field: {e.errors()[0]['loc']})"

print("✅ extract_and_validate() ready")

---
## ✍️ Step 5: Write the 3 Few-Shot Examples

These are the most important part of this notebook.
The model replicates their format, reasoning style, and field structure for every new input.

**Rule of thumb:** Pick examples that cover diverse transaction types so the model sees variety.

In [ ]:
# Each example is a (scenario, expected_JSON_dict) pair
FEW_SHOT_EXAMPLES = [
    {
        "input": (
            "XYZ Pvt Ltd, a GST-registered supplier in Maharashtra, raised a Rs 5 lakh invoice "
            "for annual maintenance services to ABC Ltd, also registered in Maharashtra."
        ),
        "output": {
            "category":             "B2B Intra-State Supply",
            "applicable_tax":       "CGST+SGST",
            "rate":                 "18% (9% CGST + 9% SGST)",
            "section":              "Section 9, CGST Act 2017",
            "reasoning":            (
                "CLASSIFY: Taxable service supply between two GST-registered entities in the same state.\n"
                "LOCATE RULE: Intra-state supply → Section 9 CGST applies; SGST collected in parallel.\n"
                "APPLY: Maintenance services fall under SAC 998712; standard rate is 18%.\n"
                "CONCLUSION: CGST @ 9% + SGST @ 9% = 18% total. No exemption applies."
            ),
            "confidence":             0.97,
            "human_review_required":  False,
        },
    },
    {
        "input": (
            "An individual freelance developer in Bengaluru provides cloud architecture consulting "
            "to a US fintech company. Invoice value: USD 3,000. The freelancer has filed an LUT "
            "with GST authorities for the current financial year."
        ),
        "output": {
            "category":             "Export of Services",
            "applicable_tax":       "Zero-Rated",
            "rate":                 "0% (LUT filed — no IGST charged)",
            "section":              "Section 16(1)(a), IGST Act 2017; Notification 37/2017-CT",
            "reasoning":            (
                "CLASSIFY: IT consulting service supplied to a foreign recipient — qualifies as export of service.\n"
                "LOCATE RULE: Export of services = zero-rated supply under Section 16 IGST Act.\n"
                "APPLY: LUT is filed, so supplier can export without charging IGST per Section 16(3)(a).\n"
                "CONCLUSION: Zero-rated. 0% IGST. Foreign currency remittance must be received within prescribed time."
            ),
            "confidence":             0.95,
            "human_review_required":  False,
        },
    },
    {
        "input": (
            "A Delhi-based retailer registered under the GST Composition Scheme (annual turnover Rs 1.1 crore) "
            "issues a bill to a walk-in customer in Delhi for Rs 25,000 worth of garments."
        ),
        "output": {
            "category":             "Composition Scheme — B2C Supply",
            "applicable_tax":       "CGST+SGST",
            "rate":                 "1% composite tax (0.5% CGST + 0.5% SGST)",
            "section":              "Section 10, CGST Act 2017; Rule 7, CGST Rules 2017",
            "reasoning":            (
                "CLASSIFY: Composition scheme dealer selling goods to a B2C customer in the same state.\n"
                "LOCATE RULE: Section 10 CGST — composition levy; rate for traders = 1% of turnover.\n"
                "APPLY: Dealer pays 1% composite tax from own pocket; customer does NOT pay GST.\n"
                "CONCLUSION: Bill of Supply must be issued (not a Tax Invoice). Dealer cannot collect GST."
            ),
            "confidence":             0.93,
            "human_review_required":  False,
        },
    },
]


def format_few_shot_block(examples):
    """Format examples as a string block to inject into the system prompt."""
    lines = []
    for i, ex in enumerate(examples, 1):
        lines.append(f"--- EXAMPLE {i} ---")
        lines.append(f"INPUT: {ex['input']}")
        lines.append(f"OUTPUT: {json.dumps(ex['output'], indent=2)}")
        lines.append("")
    return "\n".join(lines)


FEW_SHOT_BLOCK = format_few_shot_block(FEW_SHOT_EXAMPLES)

print("✅ Few-shot block ready.")
print(f"   Characters: {len(FEW_SHOT_BLOCK)}")
print("\nPreview (first 400 chars):")
print(FEW_SHOT_BLOCK[:400] + "...")

---
## 🔗 Step 6: Build the Classifier Function

Architecture:  **System prompt (few-shot + schema) → AzureOpenAI → extract_and_validate → TaxClassification**

In [ ]:
CLASSIFIER_SYSTEM = (
    "You are a senior GST consultant at a Big4 firm.\n"
    "Classify tax scenarios and return ONLY a valid JSON object.\n"
    "Do NOT include any text, explanation, or markdown before or after the JSON.\n\n"
    "=== OUTPUT SCHEMA ===\n"
    + SCHEMA_DESCRIPTION
    + "\n\n"
    + "=== EXAMPLES (follow this exact reasoning style and JSON format) ===\n"
    + FEW_SHOT_BLOCK
)


def classify_tax_scenario(scenario, verbose=True):
    """
    Classify a tax scenario using Few-Shot CoT prompting.
    Returns TaxClassification object or None on parse failure.
    """
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": CLASSIFIER_SYSTEM},
            {"role": "user",   "content": f"Classify this tax scenario:\n{scenario}"},
        ],
        temperature = 0.0,
        max_tokens  = 800,
    )

    raw = response.choices[0].message.content
    result, error = extract_and_validate(raw)

    if verbose:
        if result:
            flag = " ⚠️  HUMAN REVIEW" if result.human_review_required else ""
            print(f"  ✅  {result.category:<42} | {result.rate:<25} | conf={result.confidence:.2f}{flag}")
        else:
            print(f"  ❌  PARSE ERROR: {error}")
            print(f"     Raw response (first 200 chars): {raw[:200]}")

    return result, error


print("✅ classify_tax_scenario() ready")
print(f"   System prompt length: {len(CLASSIFIER_SYSTEM)} characters")

---
## 🧪 Step 7: Quick Smoke Test — 3 Scenarios

In [ ]:
smoke_tests = [
    "Mumbai software company sells Rs 2L SaaS license to registered client in Hyderabad.",
    "Indian company pays USD 500/month to Microsoft Ireland for Azure cloud services. No GST on invoice.",
    "A private school charges Rs 80,000 annual tuition to a student.",
]

print("Running smoke tests...\n")
for scenario in smoke_tests:
    print(f"  Scenario: {scenario[:70]}")
    classify_tax_scenario(scenario)
    print()

print("\n✅ Smoke tests complete — proceed to full run if no errors.")

---
## 🚀 Step 8: Full Run — 10 Indian Tax Scenarios

In [ ]:
TEST_SCENARIOS = [
    # 1 — B2B inter-state
    "A Mumbai-based software company sells a Rs 2 lakh annual SaaS license to a client registered in Hyderabad.",
    # 2 — Export of goods
    "An apparel exporter in Tirupur ships garments worth USD 10,000 to a UK retailer under a shipping bill.",
    # 3 — Import of services (RCM)
    "An Indian company subscribes to Microsoft Azure and pays USD 500/month to Microsoft Ireland. No GST charged on invoice.",
    # 4 — Exempt supply
    "A private school charges tuition fees of Rs 80,000 for the academic year from a student.",
    # 5 — Composition scheme
    "A Bengaluru restaurant (dine-in, AC) charges Rs 1,200 for a meal to a walk-in customer.",
    # 6 — TDS under GST (TCS by government)
    "A central government department awards a Rs 12 lakh contract to a supplier for IT infrastructure setup.",
    # 7 — ITC blocked credit
    "A company purchases a car worth Rs 15 lakh for the personal use of its director. Can it claim ITC?",
    # 8 — Reverse Charge Mechanism
    "A registered company in Chennai avails legal services from an individual advocate for Rs 50,000.",
    # 9 — E-commerce operator TCS
    "A Flipkart seller registered in Rajasthan sells goods via Flipkart to a buyer in Karnataka.",
    # 10 — Ambiguous scenario (should get low confidence)
    "A freelance consultant in Goa provides research services to a university in the US. The nature of the research is unclear — it may be academic or commercial.",
]

print(f"Running {len(TEST_SCENARIOS)} scenarios...\n")

results      = []
parse_errors = []

for i, scenario in enumerate(TEST_SCENARIOS, 1):
    print(f"Q{i:02d}: {scenario[:70]}")
    result, error = classify_tax_scenario(scenario)
    results.append(result)
    if error:
        parse_errors.append((i, error))
    print()

print("─" * 70)
print(f"Parsed successfully : {len(results) - len(parse_errors)}/{len(TEST_SCENARIOS)}")
print(f"Parse errors        : {len(parse_errors)}")

---
## 🔍 Step 9: Inspect a Specific Result in Full

In [ ]:
# Change IDX (0–9) to inspect any scenario
IDX = 2

r = results[IDX]
if r is not None:
    print(f"SCENARIO {IDX+1}:")
    print(f"  {TEST_SCENARIOS[IDX]}")
    print()
    print(f"  category              : {r.category}")
    print(f"  applicable_tax        : {r.applicable_tax}")
    print(f"  rate                  : {r.rate}")
    print(f"  section               : {r.section}")
    print(f"  confidence            : {r.confidence}")
    print(f"  human_review_required : {r.human_review_required}")
    print(f"\n  REASONING:")
    for line in r.reasoning.split("\n"):
        print(f"    {line}")
else:
    print(f"Result {IDX+1} failed to parse.")

---
## 📊 Step 10: Confidence Distribution & Human Review Flags

In [ ]:
valid = [r for r in results if r is not None]

print("CONFIDENCE DISTRIBUTION")
print("-" * 55)
for i, r in enumerate(valid):
    bar  = "█" * int(r.confidence * 20)
    flag = " ← REVIEW" if r.human_review_required else ""
    print(f"  Q{i+1:02d}  {bar:<20}  {r.confidence:.2f}{flag}")

avg_conf = sum(r.confidence for r in valid) / len(valid) if valid else 0
flagged  = sum(1 for r in valid if r.human_review_required)

print(f"\nAverage confidence   : {avg_conf:.2f}")
print(f"Flagged for review   : {flagged}/{len(valid)}")
print(f"Parse success rate   : {len(valid)}/{len(TEST_SCENARIOS)} = {len(valid)/len(TEST_SCENARIOS)*100:.0f}%")

if flagged > 0:
    print("\n⚠️  Scenarios requiring human review:")
    for i, r in enumerate(valid):
        if r.human_review_required:
            print(f"  Q{i+1:02d}: {TEST_SCENARIOS[i][:80]}")

---
## 💾 Step 11: Export Results to JSON

In [ ]:
export_rows = []
for i, (scenario, result) in enumerate(zip(TEST_SCENARIOS, results), 1):
    if result is not None:
        export_rows.append({
            "id":                    i,
            "scenario":              scenario,
            "category":              result.category,
            "applicable_tax":        result.applicable_tax,
            "rate":                  result.rate,
            "section":               result.section,
            "reasoning":             result.reasoning,
            "confidence":            result.confidence,
            "human_review_required": result.human_review_required,
        })
    else:
        export_rows.append({"id": i, "scenario": scenario, "error": "parse_failed"})

payload = {
    "notebook":    "Day3B_FewShot_Classifier",
    "exported_at": datetime.datetime.now().isoformat(),
    "model":       AZURE_DEPLOYMENT_NAME,
    "results":     export_rows,
}

with open("day3b_results.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print(f"✅ Results saved to day3b_results.json")
print(f"   Rows exported: {len(export_rows)}")

---
## 🎓 What This Notebook Demonstrated

| Capability | Status |
|---|---|
| Few-shot examples lock the output format | ✅ |
| Pydantic validates every response — bad data never reaches the pipeline | ✅ |
| Confidence gating auto-flags ambiguous cases | ✅ |
| Output is machine-readable — no manual parsing | ✅ |
| Full reasoning trace preserved for auditability | ✅ |

**The output from this classifier can feed directly into:**
- A tax compliance database (typed fields, schema-validated)
- A workflow router (`human_review_required` gates escalation)
- Day 4's structured extraction pipeline (same pattern, applied to unstructured documents)

---
## ➡️ Next: Notebook 3C — Head-to-Head Benchmark

We run all four prompting strategies (Zero-Shot, Zero-Shot CoT, Few-Shot, Few-Shot CoT) on the **same 10 cases** with ground-truth labels, and produce a quantified comparison table + chart.